# UK Retail Sales: Time-Series Forecasting

This notebook develops and evaluates models for forecasting year-on-year UK retail sales-volume growth.

## Objectives

- Prepare the monthly time series for modelling.
- Examine trend, seasonality and stationarity.
- Create a time-based training and test split.
- Establish a naïve baseline forecast.
- Compare exponential-smoothing and SARIMA models.
- Evaluate forecasts using MAE and RMSE.
- Produce a 12-month forecast with uncertainty intervals.

## Target variable

The target is `volume_yoy_pct`, representing the annual percentage change in the estimated quantity of retail goods purchased. This provides a clearer measure of underlying consumer demand than nominal sales value because it removes the effect of changing prices.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from sklearn.metrics import mean_absolute_error, mean_squared_error
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.stattools import adfuller

In [ ]:
sns.set_theme(style="whitegrid")

plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["axes.titleweight"] = "bold"

In [ ]:
sns.set_theme(style="whitegrid")

plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["axes.titleweight"] = "bold"

In [ ]:
data_path = Path("..") / "data" / "processed" / "retail_sales_core_monthly.csv"

retail = pd.read_csv(data_path, parse_dates=["date"])

volume_growth = (
    retail
    .set_index("date")["volume_yoy_pct"]
    .asfreq("MS")
)

print("File found:", data_path.exists())
print("Observations:", len(volume_growth))
print("Missing values:", volume_growth.isna().sum())
print("Start date:", volume_growth.index.min())
print("End date:", volume_growth.index.max())

volume_growth.head()

## 1. Time-series preparation

The forecasting series contains 356 continuous monthly observations from January 1997 to August 2026, with no missing values. The data has been indexed by date and assigned a monthly-start frequency (`MS`), which enables statsmodels to recognise the regular time intervals.

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))

volume_growth.plot(
    ax=ax,
    color="darkorange",
    linewidth=1.8
)

ax.axhline(0, color="black", linewidth=1, linestyle="--")
ax.set_title("UK Retail Sales-Volume Growth")
ax.set_xlabel("Date")
ax.set_ylabel("Year-on-year change (%)")

plt.tight_layout()
plt.show()

## 2. Time-series structure

The series is generally centred around modest positive growth, but it contains an exceptional structural disruption during 2020 and 2021. The sharp fall and rebound reflect COVID-19 restrictions, reopening and year-on-year base effects.

The prolonged negative growth during 2022 and 2023 reflects the reduction in retail purchasing volumes during the cost-of-living crisis. More recent observations suggest a recovery toward positive growth.

These structural changes and extreme observations may reduce forecast accuracy and must be considered when evaluating the models.

In [ ]:
decomposition = seasonal_decompose(
    volume_growth,
    model="additive",
    period=12,
    extrapolate_trend="freq"
)

fig = decomposition.plot()
fig.set_size_inches(14, 10)
fig.suptitle(
    "Decomposition of UK Retail Sales-Volume Growth",
    fontsize=16,
    fontweight="bold"
)

plt.tight_layout()
plt.show()

### Decomposition finding

The decomposition indicates that long-term economic movements and exceptional shocks are more important than seasonality for explaining annual retail sales-volume growth.

The estimated seasonal component is very small, which is expected because the ONS series is already seasonally adjusted and measured as a year-on-year percentage change. In contrast, the residual component contains extreme observations during 2020 and 2021, confirming that the pandemic created movements that normal trend and seasonal patterns could not explain.

In [ ]:
adf_result = adfuller(volume_growth)

adf_summary = pd.Series(
    {
        "ADF statistic": adf_result[0],
        "p-value": adf_result[1],
        "Lags used": adf_result[2],
        "Observations used": adf_result[3],
        "1% critical value": adf_result[4]["1%"],
        "5% critical value": adf_result[4]["5%"],
        "10% critical value": adf_result[4]["10%"]
    }
).round(4)

adf_summary

### Stationarity finding

The Augmented Dickey–Fuller test produced a p-value of 0.0008. This is below the 5% significance threshold, so the null hypothesis of a unit root is rejected.

The year-on-year sales-volume growth series is therefore treated as stationary, and ordinary differencing is not initially required. However, the extreme pandemic observations and structural economic changes may still affect model performance.

## 3. Training and test split

The final 24 months are reserved as unseen test data. Models will be fitted only on the earlier observations and evaluated by forecasting the test period.

A time-based split is essential because randomly shuffling time-series data would allow future information to influence model training.

In [ ]:
test_size = 24

train = volume_growth.iloc[:-test_size]
test = volume_growth.iloc[-test_size:]

print("Training observations:", len(train))
print("Training period:", train.index.min(), "to", train.index.max())

print("\nTest observations:", len(test))
print("Test period:", test.index.min(), "to", test.index.max())

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))

train.plot(
    ax=ax,
    label="Training data",
    color="steelblue"
)

test.plot(
    ax=ax,
    label="Test data",
    color="darkorange"
)

ax.axvline(
    test.index.min(),
    color="black",
    linestyle="--",
    label="Forecast start"
)

ax.axhline(0, color="grey", linewidth=1)
ax.set_title("Training and Test Split")
ax.set_xlabel("Date")
ax.set_ylabel("Year-on-year volume growth (%)")
ax.legend()

plt.tight_layout()
plt.show()

## 4. Naïve baseline forecast

A naïve forecast assumes that every future observation will equal the final value in the training data. Although simple, it provides an essential benchmark: a more complex model is useful only if it produces lower forecast errors than this baseline.

In [ ]:
naive_forecast = pd.Series(
    train.iloc[-1],
    index=test.index,
    name="naive_forecast"
)

naive_mae = mean_absolute_error(test, naive_forecast)
naive_rmse = np.sqrt(mean_squared_error(test, naive_forecast))

print("Last training value:", round(train.iloc[-1], 2))
print("Naïve MAE:", round(naive_mae, 2))
print("Naïve RMSE:", round(naive_rmse, 2))

## 5. Exponential-smoothing model

The exponential-smoothing model allows the underlying level and trend to change over time while giving greater weight to recent observations. A damped trend is used so that the model's projected growth does not continue increasing or decreasing indefinitely.

In [ ]:
ets_model = ExponentialSmoothing(
    train,
    trend="add",
    damped_trend=True,
    seasonal=None,
    initialization_method="estimated"
).fit(optimized=True)

ets_forecast = ets_model.forecast(len(test))

ets_mae = mean_absolute_error(test, ets_forecast)
ets_rmse = np.sqrt(mean_squared_error(test, ets_forecast))

print("Exponential-smoothing MAE:", round(ets_mae, 2))
print("Exponential-smoothing RMSE:", round(ets_rmse, 2))

## 6. SARIMA model

A set of SARIMA specifications is compared using the Akaike Information Criterion (AIC). AIC balances model fit against complexity, with a lower value indicating a preferable model among the candidates.

Because the series is stationary and has very limited seasonality, no ordinary or seasonal differencing is applied. The search considers small autoregressive and moving-average orders to avoid unnecessary complexity.

In [ ]:
import warnings

candidate_results = []
best_aic = np.inf
best_sarima_fit = None
best_order = None
best_seasonal_order = None

with warnings.catch_warnings():
    warnings.simplefilter("ignore")

    for p in range(3):
        for q in range(3):
            for P in range(2):
                for Q in range(2):
                    order = (p, 0, q)
                    seasonal_order = (P, 0, Q, 12)

                    try:
                        model = SARIMAX(
                            train,
                            order=order,
                            seasonal_order=seasonal_order,
                            trend="c",
                            enforce_stationarity=False,
                            enforce_invertibility=False
                        )

                        fitted_model = model.fit(disp=False)

                        candidate_results.append(
                            {
                                "order": order,
                                "seasonal_order": seasonal_order,
                                "AIC": fitted_model.aic
                            }
                        )

                        if fitted_model.aic < best_aic:
                            best_aic = fitted_model.aic
                            best_sarima_fit = fitted_model
                            best_order = order
                            best_seasonal_order = seasonal_order

                    except Exception:
                        continue

sarima_candidates = (
    pd.DataFrame(candidate_results)
    .sort_values("AIC")
    .reset_index(drop=True)
)

print("Best order:", best_order)
print("Best seasonal order:", best_seasonal_order)
print("Best AIC:", round(best_aic, 2))

sarima_candidates.head()

In [ ]:
sarima_test_result = best_sarima_fit.get_forecast(
    steps=len(test)
)

sarima_forecast = sarima_test_result.predicted_mean

sarima_mae = mean_absolute_error(test, sarima_forecast)
sarima_rmse = np.sqrt(mean_squared_error(test, sarima_forecast))

print("SARIMA MAE:", round(sarima_mae, 2))
print("SARIMA RMSE:", round(sarima_rmse, 2))

In [ ]:
model_comparison = pd.DataFrame(
    {
        "Model": [
            "Naïve",
            "Exponential smoothing",
            "SARIMA"
        ],
        "MAE": [
            naive_mae,
            ets_mae,
            sarima_mae
        ],
        "RMSE": [
            naive_rmse,
            ets_rmse,
            sarima_rmse
        ]
    }
).sort_values("RMSE").reset_index(drop=True)

model_comparison.round(2)

In [ ]:
fig, ax = plt.subplots(figsize=(14, 7))

train.tail(24).plot(
    ax=ax,
    label="Recent training data",
    color="grey",
    linewidth=2
)

test.plot(
    ax=ax,
    label="Actual test data",
    color="black",
    linewidth=2.5
)

naive_forecast.plot(
    ax=ax,
    label="Naïve forecast",
    linestyle="--"
)

ets_forecast.plot(
    ax=ax,
    label="Exponential smoothing",
    linestyle="--"
)

sarima_forecast.plot(
    ax=ax,
    label="SARIMA",
    linestyle="--"
)

ax.axvline(
    test.index.min(),
    color="black",
    linestyle=":",
    label="Forecast start"
)

ax.axhline(0, color="grey", linewidth=1)
ax.set_title("Model Forecasts Compared with Actual Retail-Volume Growth")
ax.set_xlabel("Date")
ax.set_ylabel("Year-on-year volume growth (%)")
ax.legend()

plt.tight_layout()
plt.show()

### Model-evaluation finding

The naïve model achieved the lowest test errors, with an MAE of 1.17 percentage points and an RMSE of 1.50 percentage points. Exponential smoothing performed almost identically but did not improve upon the benchmark.

SARIMA produced larger forecast fluctuations and the highest errors. Although it fitted the training data using autoregressive, moving-average and annual seasonal relationships, these historical patterns did not generalise reliably to the recent test period.

The naïve model is therefore selected for the final forecast. This demonstrates that a transparent, simple model can outperform more complex approaches when a time series is affected by structural economic changes and irregular shocks.

## 7. Final 12-month forecast

The selected naïve model is refitted conceptually using all available observations. It forecasts that annual retail sales-volume growth will remain equal to the latest observed rate.

An approximate 95% uncertainty range is calculated using the naïve model's test RMSE. This interval communicates forecast risk but should not be interpreted as a guarantee, particularly because unexpected economic shocks cannot be predicted from historical data.

In [ ]:
forecast_horizon = 12

future_dates = pd.date_range(
    start=volume_growth.index.max() + pd.offsets.MonthBegin(1),
    periods=forecast_horizon,
    freq="MS"
)

forecast_value = volume_growth.iloc[-1]
uncertainty_margin = 1.96 * naive_rmse

final_forecast = pd.DataFrame(
    {
        "date": future_dates,
        "forecast_volume_yoy_pct": forecast_value,
        "lower_95_pct": forecast_value - uncertainty_margin,
        "upper_95_pct": forecast_value + uncertainty_margin,
        "model": "Naïve"
    }
)

final_forecast_display = final_forecast.copy()

numeric_columns = [
    "forecast_volume_yoy_pct",
    "lower_95_pct",
    "upper_95_pct"
]

final_forecast_display[numeric_columns] = (
    final_forecast_display[numeric_columns].round(2)
)

final_forecast_display

In [ ]:
history = volume_growth.tail(36)

history_dates = history.index.to_numpy()
forecast_dates = final_forecast["date"].to_numpy()

fig, ax = plt.subplots(figsize=(14, 7))

# Historical data
ax.plot(
    history_dates,
    history.to_numpy(),
    color="steelblue",
    linewidth=2,
    label="Historical volume growth"
)

# Forecast
ax.plot(
    forecast_dates,
    final_forecast["forecast_volume_yoy_pct"].to_numpy(),
    color="darkorange",
    linewidth=2,
    marker="o",
    label="12-month forecast"
)

# Approximate uncertainty range
ax.fill_between(
    forecast_dates,
    final_forecast["lower_95_pct"].to_numpy(),
    final_forecast["upper_95_pct"].to_numpy(),
    color="orange",
    alpha=0.2,
    label="Approximate 95% uncertainty range"
)

# Forecast starting point
ax.axvline(
    final_forecast["date"].min(),
    color="black",
    linestyle="--",
    linewidth=1.5,
    label="Forecast start"
)

# Zero-growth reference line
ax.axhline(
    0,
    color="grey",
    linewidth=1
)

# Ensure the forecast period is visible
ax.set_xlim(
    history.index.min(),
    final_forecast["date"].max()
)

ax.set_title("UK Retail Sales Volume Growth: 12-Month Forecast")
ax.set_xlabel("Date")
ax.set_ylabel("Year-on-year volume growth (%)")
ax.legend()
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
forecast_path = (
    Path("..")
    / "data"
    / "processed"
    / "retail_sales_volume_forecast.csv"
)

final_forecast.to_csv(
    forecast_path,
    index=False
)

print("Forecast saved to:", forecast_path)
print("Rows exported:", len(final_forecast))

## Conclusions

This analysis compared three forecasting approaches for UK retail sales volume growth: a naïve baseline, exponential smoothing and SARIMA.

The naïve model achieved the best test-set performance, with an MAE of 1.17 percentage points and an RMSE of 1.50 percentage points. Exponential smoothing produced very similar results, while SARIMA performed less accurately.

The selected naïve model forecasts year-on-year retail sales volume growth of 2.4% from September 2026 to August 2027. The approximate 95% uncertainty range extends from -0.5% to 5.3%, indicating that both modest contraction and stronger growth remain possible.

The forecast should be treated as a simple baseline rather than a precise prediction. It assumes that the latest observed growth rate continues and does not account for economic variables such as inflation, interest rates, consumer confidence or unexpected structural shocks.

The forecast data has been exported for use in the Tableau dashboard.